In [1]:
import pandas as pd
import numpy as np
import torch
import transformers
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoModel, BertTokenizerFast, DistilBertTokenizerFast, AutoTokenizer
from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from torch.utils.data import Dataset
import torch.nn as nn
from sklearn.utils.class_weight import compute_class_weight
from tqdm.notebook import tqdm
from nltk.tokenize import sent_tokenize, word_tokenize
import os
import nltk
from sklearn.feature_extraction.text import CountVectorizer
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Read Data

In [33]:
os.chdir('/content/drive/My Drive/NLP-Letters-V2/notebooks/')
train_df = pd.read_csv('../data/combined_letters_degendered_with_topics_train.csv')
val_df = pd.read_csv('../data/combined_letters_degendered_with_topics_val.csv')
test_df = pd.read_csv('../data/combined_letters_degendered_with_topics_test.csv')

# Create Training and Test Sets

In [34]:
bert = AutoModel.from_pretrained("distilbert/distilbert-base-uncased")
tokenizer = AutoTokenizer.from_pretrained("distilbert/distilbert-base-uncased")

In [35]:
class TextWithTopicsDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.text_data = df['full_text'].tolist()
        self.topic_features = df.filter(like='topic_').values
        self.labels = df['label'].values
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.text_data)

    def __getitem__(self, idx):
        text = self.text_data[idx]
        topics = torch.tensor(self.topic_features[idx], dtype=torch.float)
        label = torch.tensor(self.labels[idx], dtype=torch.long)

        tokens = self.tokenizer(
            text,
            padding='max_length',
            truncation=True,
            return_tensors="pt"
        )

        return {
            'input_ids': tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0),
            'topic_feats': topics,
            'label': label
        }

In [36]:
train_dataset = TextWithTopicsDataset(train_df, tokenizer)
val_dataset = TextWithTopicsDataset(val_df, tokenizer)
test_dataset = TextWithTopicsDataset(test_df, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
test_loader = DataLoader(test_dataset, batch_size=16)

# Create Model

In [37]:
class BERTWithTopics(nn.Module):
    def __init__(self, bert, topic_feat_dim, num_classes):
        super(BERTWithTopics, self).__init__()
        self.bert = bert
        self.hidden_size = self.bert.config.hidden_size
        self.topic_feat_dim = topic_feat_dim
        self.dropout = nn.Dropout(0.1)
        self.relu = nn.ReLU()

        # Attention projection layers
        self.query_proj = nn.Linear(topic_feat_dim, self.hidden_size)
        self.key_proj = nn.Linear(self.hidden_size, self.hidden_size)
        self.value_proj = nn.Linear(self.hidden_size, self.hidden_size)

        # Classifier
        self.fc1 = nn.Linear(self.hidden_size + topic_feat_dim, 128)
        self.fc2 = nn.Linear(128, num_classes)
        self.softmax = nn.LogSoftmax(dim=1)  # For classification; use NLLLoss

    def forward(self, input_ids, attention_mask, topic_feats):
      # BERT forward pass
      bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
      token_embeddings = bert_out.last_hidden_state  # [batch_size, seq_len, hidden_size]

      # Attention: topic_feats as query, LLM output as key/value
      Q = self.query_proj(topic_feats).unsqueeze(1)                  # [batch_size, 1, hidden_size]
      K = self.key_proj(token_embeddings)                            # [batch_size, seq_len, hidden_size]
      V = self.value_proj(token_embeddings)                          # [batch_size, seq_len, hidden_size]

      attn_scores = torch.bmm(Q, K.transpose(1, 2)) / (self.hidden_size ** 0.5)
      attn_weights = torch.softmax(attn_scores, dim=-1)              # [batch_size, 1, seq_len]
      context = torch.bmm(attn_weights, V).squeeze(1)                # [batch_size, hidden_size]

      # Concatenate context vector with topic vector
      x = torch.cat((context, topic_feats), dim=1)                   # [batch_size, hidden + topic_dim]
      x = self.dropout(self.relu(self.fc1(x)))
      x = self.fc2(x)
      return self.softmax(x)

In [38]:
model = BERTWithTopics(bert, 97, 2)
model = model.to(device)

In [39]:
# Freeze all BERT layers
for param in model.bert.parameters():
    param.requires_grad = False

# Unfreeze only the last 2 layers of DistilBERT
for layer in model.bert.transformer.layer[-2:]:
    for param in layer.parameters():
        param.requires_grad = True

In [40]:
batch_size = 16
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)
class_weights = compute_class_weight(class_weight='balanced', classes=np.unique(train_df['label']), y=train_df['label'])
weights= torch.tensor(class_weights,dtype=torch.float)
weights = weights.to(device)
cross_entropy  = nn.NLLLoss(weight=weights)
epochs = 10

In [41]:
# function to train the model
def train():

  model.train()

  total_loss, total_accuracy = 0, 0

  # empty list to save model predictions
  total_preds=[]
  total_labels=[]

  # iterate over batches
  for step,batch in enumerate(tqdm(train_loader, desc="Training", leave=True)):

    # # progress update after every 50 batches.
    # if step % 50 == 0 and not step == 0:
    #   print('  Batch {:>5,}  of  {:>5,}.'.format(step, len(train_dataloader)))

    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    topic_feats = batch['topic_feats'].to(device)
    labels = batch['label'].to(device)

    # clear previously calculated gradients
    model.zero_grad()

    # get model predictions for the current batch
    preds = model(input_ids, attention_mask, topic_feats)

    # compute the loss between actual and predicted values
    loss = cross_entropy(preds, labels)

    # add on to the total loss
    total_loss = total_loss + loss.item()

    # backward pass to calculate the gradients
    loss.backward()

    # clip the the gradients to 1.0. It helps in preventing the exploding gradient problem
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

    # update parameters
    optimizer.step()

    # model predictions are stored on GPU. So, push it to CPU
    preds=preds.detach().cpu().numpy()

    # append the model predictions
    total_preds.append(preds)

    labels=labels.detach().cpu().numpy()

    # append labels
    total_labels.append(labels)

  # compute the training loss of the epoch
  avg_loss = total_loss / len(train_loader)

  # predictions are in the form of (no. of batches, size of batch, no. of classes).
  # reshape the predictions in form of (number of samples, no. of classes)
  total_preds  = np.concatenate(total_preds, axis=0)

  total_labels = np.concatenate(total_labels, axis=0)

  epoch_preds = np.argmax(total_preds, axis = 1)

  print('Training Classification Report: \n', classification_report(total_labels, epoch_preds))
  print('Training Confusion Matrix: \n', confusion_matrix(total_labels, epoch_preds))

  #returns the loss and predictions
  return avg_loss, total_preds

In [42]:
# function for evaluating the model
def evaluate():

  print("\nEvaluating...")

  # deactivate dropout layers
  model.eval()

  total_loss, total_accuracy = 0, 0

  # empty list to save the model predictions
  total_preds = []
  total_labels = []

  # iterate over batches
  for step,batch in enumerate(val_loader):

    # # Progress update every 50 batches.
    # if step % 50 == 0 and not step == 0:

    #   # Calculate elapsed time in minutes.
    #   elapsed = format_time(time.time() - t0)

    #   # Report progress.
    #   print('  Batch {:>5,}  of  {:>5,}.'.format(step, len(val_dataloader)))

    input_ids = batch['input_ids'].to(device)
    attention_mask = batch['attention_mask'].to(device)
    topic_feats = batch['topic_feats'].to(device)
    labels = batch['label'].to(device)

    # deactivate autograd
    with torch.no_grad():

      # model predictions
      preds = model(input_ids, attention_mask, topic_feats)

      # compute the validation loss between actual and predicted values
      loss = cross_entropy(preds,labels)

      total_loss = total_loss + loss.item()

      preds = preds.detach().cpu().numpy()

      total_preds.append(preds)

      labels = labels.detach().cpu().numpy()

      total_labels.append(labels)

  # compute the validation loss of the epoch
  avg_loss = total_loss / len(val_loader)

  # reshape the predictions in form of (number of samples, no. of classes)
  total_preds  = np.concatenate(total_preds, axis=0)

  total_labels = np.concatenate(total_labels, axis=0)

  epoch_preds = np.argmax(total_preds, axis = 1)

  print('Validation Classification Report: \n', classification_report(total_labels, epoch_preds))
  print('Validation Confusion Matrix: \n', confusion_matrix(total_labels, epoch_preds))

  return avg_loss, total_preds

In [43]:
# set initial loss to infinite
best_valid_loss = float('inf')

# empty lists to store training and validation loss of each epoch
train_losses=[]
valid_losses=[]

#for each epoch
for epoch in tqdm(range(epochs), desc="Training"):

    print('\n Epoch {:} / {:}'.format(epoch + 1, epochs))

    #train model
    train_loss, _ = train()

    #evaluate model
    valid_loss, _ = evaluate()

    #save the best model
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        print('Model Saved!')
        torch.save(model, '../saved_models/saved_model.pt')

    # append training and validation loss
    train_losses.append(train_loss)
    valid_losses.append(valid_loss)

    print(f'\nTraining Loss: {train_loss:.3f}')
    print(f'Validation Loss: {valid_loss:.3f}')

Training:   0%|          | 0/10 [00:00<?, ?it/s]


 Epoch 1 / 10


Training:   0%|          | 0/404 [00:00<?, ?it/s]

Training Classification Report: 
               precision    recall  f1-score   support

           0       0.32      0.17      0.22      2006
           1       0.69      0.84      0.76      4457

    accuracy                           0.63      6463
   macro avg       0.51      0.51      0.49      6463
weighted avg       0.58      0.63      0.59      6463

Training Confusion Matrix: 
 [[ 342 1664]
 [ 713 3744]]

Evaluating...
Validation Classification Report: 
               precision    recall  f1-score   support

           0       0.32      0.85      0.47       223
           1       0.74      0.19      0.30       496

    accuracy                           0.40       719
   macro avg       0.53      0.52      0.39       719
weighted avg       0.61      0.40      0.35       719

Validation Confusion Matrix: 
 [[190  33]
 [401  95]]
Model Saved!

Training Loss: 0.693
Validation Loss: 0.692

 Epoch 2 / 10


Training:   0%|          | 0/404 [00:00<?, ?it/s]

...

Training Classification Report: 
               precision    recall  f1-score   support

           0       0.48      0.65      0.55      2006
           1       0.81      0.69      0.74      4457

    accuracy                           0.68      6463
   macro avg       0.65      0.67      0.65      6463
weighted avg       0.71      0.68      0.69      6463

Training Confusion Matrix: 
 [[1307  699]
 [1398 3059]]

Evaluating...
Validation Classification Report: 
               precision    recall  f1-score   support

           0       0.42      0.40      0.41       223
           1       0.74      0.75      0.74       496

    accuracy                           0.64       719
   macro avg       0.58      0.58      0.58       719
weighted avg       0.64      0.64      0.64       719

Validation Confusion Matrix: 
 [[ 89 134]
 [123 373]]

Training Loss: 0.608
Validation Loss: 0.701

 Epoch 9 / 10


Training:   0%|          | 0/404 [00:00<?, ?it/s]

Training Classification Report: 
               precision    recall  f1-score   support

           0       0.50      0.68      0.58      2006
           1       0.83      0.70      0.76      4457

    accuracy                           0.69      6463
   macro avg       0.67      0.69      0.67      6463
weighted avg       0.73      0.69      0.70      6463

Training Confusion Matrix: 
 [[1363  643]
 [1337 3120]]

Evaluating...
Validation Classification Report: 
               precision    recall  f1-score   support

           0       0.44      0.40      0.42       223
           1       0.74      0.77      0.75       496

    accuracy                           0.66       719
   macro avg       0.59      0.58      0.59       719
weighted avg       0.65      0.66      0.65       719

Validation Confusion Matrix: 
 [[ 89 134]
 [114 382]]

Training Loss: 0.587
Validation Loss: 0.708

 Epoch 10 / 10


Training:   0%|          | 0/404 [00:00<?, ?it/s]

Training Classification Report: 
               precision    recall  f1-score   support

           0       0.53      0.69      0.60      2006
           1       0.84      0.72      0.77      4457

    accuracy                           0.71      6463
   macro avg       0.68      0.71      0.69      6463
weighted avg       0.74      0.71      0.72      6463

Training Confusion Matrix: 
 [[1387  619]
 [1252 3205]]

Evaluating...
Validation Classification Report: 
               precision    recall  f1-score   support

           0       0.36      0.65      0.46       223
           1       0.75      0.47      0.58       496

    accuracy                           0.53       719
   macro avg       0.56      0.56      0.52       719
weighted avg       0.63      0.53      0.55       719

Validation Confusion Matrix: 
 [[146  77]
 [261 235]]

Training Loss: 0.564
Validation Loss: 0.736


# Test Model

In [44]:
model = torch.load('../saved_models/saved_model.pt', weights_only=False)

In [45]:

model.eval()  # Set model to eval mode

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        topic_feats = batch['topic_feats'].to(device)
        labels = batch['label'].to(device)

        outputs = model(input_ids, attention_mask, topic_feats)

        # Get predicted class (as indices)
        preds = torch.argmax(outputs, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

In [46]:
  print('Test Classification Report: \n', classification_report(all_labels, all_preds))
  print('Test Confusion Matrix: \n', confusion_matrix(all_labels, all_preds))

Test Classification Report: 
               precision    recall  f1-score   support

           0       0.38      0.60      0.47       557
           1       0.76      0.56      0.65      1241

    accuracy                           0.58      1798
   macro avg       0.57      0.58      0.56      1798
weighted avg       0.64      0.58      0.59      1798

Test Confusion Matrix: 
 [[336 221]
 [541 700]]
